# End-to-End Quadruped Stand Stabilization with MuJoCo + RL

This notebook implements a **terrain-robust stand stabilization controller** for a Unitree A1 quadruped using:
- **MuJoCo** (physics simulation via `mujoco` Python API)
- **Stable Baselines3** (PPO/SAC RL training)
- **Gymnasium** (environment interface)

---

## What we borrow from `gmaymon3/robot_quadruped_mpc_controller`

| Reference file | What is reused / adapted | What is rewritten |
|---|---|---|
| `a1_description/urdf/a1.urdf` | Joint names, link structure, mass/inertia of body + legs | Converted to inline MuJoCo MJCF XML (no mesh files needed) |
| `a1_description/xacro/stairs.xacro` | Stair geometry: length=0.64 m, width=0.31 m, height=0.17 m; recursive step pattern | Re-expressed as MuJoCo `<geom type='box'>` elements in Python |
| `a1_description/config/robot_control.yaml` | PD gains per joint group (hip P=100 D=5; thigh/calf P=300 D=8) | Translated to MuJoCo actuator `kp` / `kv` parameters |
| `a1_description/xacro/gazebo.xacro` | IMU + foot contact sensor concept | Replaced by MuJoCo `sensor` elements and direct `mjData` reads |
| `a1_description/xacro/leg.xacro` | 3-DOF leg structure (hip-thigh-calf) × 4 legs | Rebuilt as MJCF body tree |
| `slprj/` (MATLAB Simulink artifacts) | **Not reused** — MATLAB-specific generated code | Entire controller replaced by PPO/SAC policy |

---

## Notebook structure
1. Install dependencies
2. Build A1 MJCF model (inline XML, no external mesh files)
3. Procedural terrain generation (flat / stairs / slope / rocks)
4. Custom `gymnasium.Env` with obs/action space, reward, domain randomization
5. RL setup (PPO via SB3)
6. Training loop (Jupyter-friendly progress bar)
7. Evaluation + video recording

## 1. Install dependencies

Run this cell once, then **restart the kernel** before proceeding.

In [ ]:
import subprocess, sys

def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *pkgs])

pip('mujoco>=3.1.0')
pip('gymnasium>=0.29.0')
pip('stable-baselines3[extra]>=2.3.0')
pip('imageio[ffmpeg]')        # video recording
pip('tqdm', 'matplotlib', 'numpy')
print('All packages installed. Restart the kernel if this is the first run.')

## 2. Build the A1 MJCF model (inline XML)

The Unitree A1 has:
- **12 actuated joints**: 3 per leg (hip, thigh, calf) × 4 legs (FL, FR, RL, RR)
- Hip joint limits roughly ±0.8 rad; thigh ~±1.0 rad; calf ~1.0–2.7 rad

This XML is a **simplified capsule-based proxy model** — it captures the correct kinematics
and approximate masses from the reference URDF without requiring proprietary mesh files.
The PD gains come directly from `a1_description/config/robot_control.yaml`.

In [ ]:
import numpy as np

# ---------------------------------------------------------------------------
# A1 physical constants (sourced from reference repo URDF / datasheet)
# ---------------------------------------------------------------------------
BODY_MASS   = 6.0    # kg  (torso)
THIGH_MASS  = 1.013
CALF_MASS   = 0.166
HIP_MASS    = 0.696

# Stair geometry from stairs.xacro
STAIR_LENGTH = 0.640
STAIR_WIDTH  = 0.310
STAIR_HEIGHT = 0.170

# PD gains from robot_control.yaml
HIP_KP, HIP_KD     = 100.0, 5.0
THIGH_KP, THIGH_KD = 300.0, 8.0
CALF_KP, CALF_KD   = 300.0, 8.0

# Default standing pose (radians) – approximate nominal from Unitree A1 SDK
DEFAULT_POSE = np.array([
    0.0, 0.8, -1.6,   # FL hip, thigh, calf
    0.0, 0.8, -1.6,   # FR hip, thigh, calf
    0.0, 0.8, -1.6,   # RL hip, thigh, calf
    0.0, 0.8, -1.6,   # RR hip, thigh, calf
], dtype=np.float32)

print('Constants loaded. Body mass:', BODY_MASS, 'kg')
print('Default joint pose:', DEFAULT_POSE)

In [ ]:
# ---------------------------------------------------------------------------
# Helper: build a single leg XML snippet
# prefix: one of FL, FR, RL, RR
# hip_xyz: position of hip joint in body frame
# hip_side: +1 (left) or -1 (right) for axis flipping
# ---------------------------------------------------------------------------
def leg_xml(prefix, hip_xyz, hip_side=1):
    hx, hy, hz = hip_xyz
    return f"""
        <!-- {prefix} leg -->
        <body name="{prefix}_hip" pos="{hx} {hy} {hz}">
          <joint name="{prefix}_hip_joint" type="hinge" axis="1 0 0"
                 range="-0.802 0.802" damping="{HIP_KD}" stiffness="0"/>
          <geom type="capsule" fromto="0 0 0  0 {0.08*hip_side} 0"
                size="0.046" mass="{HIP_MASS}"/>
          <body name="{prefix}_thigh" pos="0 {0.083*hip_side} 0">
            <joint name="{prefix}_thigh_joint" type="hinge" axis="0 1 0"
                   range="-1.047 4.189" damping="{THIGH_KD}" stiffness="0"/>
            <geom type="capsule" fromto="0 0 0  0 0 -0.2"
                  size="0.0265" mass="{THIGH_MASS}"/>
            <body name="{prefix}_calf" pos="0 0 -0.2">
              <joint name="{prefix}_calf_joint" type="hinge" axis="0 1 0"
                     range="-2.697 -0.916" damping="{CALF_KD}" stiffness="0"/>
              <geom type="capsule" fromto="0 0 0  0 0 -0.2"
                    size="0.0265" mass="{CALF_MASS}"/>
              <!-- foot site for contact sensing -->
              <site name="{prefix}_foot" pos="0 0 -0.2" size="0.02"/>
            </body>
          </body>
        </body>"""


def build_a1_xml(terrain_xml: str = "") -> str:
    """
    Build a complete MuJoCo MJCF XML string for the A1 quadruped.

    Parameters
    ----------
    terrain_xml : str
        Extra <geom> elements placed inside the world body (terrain objects).
    """
    legs = (
        leg_xml('FL', ( 0.183,  0.047, 0),  hip_side= 1) +
        leg_xml('FR', ( 0.183, -0.047, 0),  hip_side=-1) +
        leg_xml('RL', (-0.183,  0.047, 0),  hip_side= 1) +
        leg_xml('RR', (-0.183, -0.047, 0),  hip_side=-1)
    )

    # Position targets (position servos) for each joint.
    # gear = kp from robot_control.yaml;  kv (velocity damping) set separately.
    actuators = ''
    for prefix in ['FL', 'FR', 'RL', 'RR']:
        actuators += f"""
        <position name="{prefix}_hip_act"   joint="{prefix}_hip_joint"   kp="{HIP_KP}"/>
        <position name="{prefix}_thigh_act" joint="{prefix}_thigh_joint" kp="{THIGH_KP}"/>
        <position name="{prefix}_calf_act"  joint="{prefix}_calf_joint"  kp="{CALF_KP}"/>"""

    # Sensors: IMU-like (gyro + accelerometer on trunk) + framequat for orientation
    sensors = """
        <gyro      name="imu_gyro"  site="imu_site"/>
        <accelerometer name="imu_accel" site="imu_site"/>
        <framequat name="trunk_quat" objtype="body" objname="trunk"/>
        <framelinvel name="trunk_linvel" objtype="body" objname="trunk"/>
        <frameangvel name="trunk_angvel" objtype="body" objname="trunk"/>"""

    xml = f"""<?xml version="1.0"?>
<mujoco model="a1_simplified">

  <!-- ====== Compiler & options ====== -->
  <compiler angle="radian" coordinate="local" inertiafromgeom="true"/>
  <option gravity="0 0 -9.81" timestep="0.002" integrator="RK4"/>

  <!-- ====== Defaults (PD position control) ====== -->
  <default>
    <joint armature="0.01" limited="true"/>
    <geom contype="1" conaffinity="1" condim="3"
          friction="0.8 0.02 0.001" rgba="0.8 0.6 0.4 1"/>
    <motor ctrllimited="true" ctrlrange="-1 1"/>
  </default>

  <!-- ====== Assets ====== -->
  <asset>
    <texture type="skybox" builtin="gradient" rgb1="0.3 0.5 0.7" rgb2="0 0 0"
             width="512" height="512"/>
    <texture name="grid" type="2d" builtin="checker" rgb1="0.1 0.2 0.3"
             rgb2="0.2 0.3 0.4" width="300" height="300"/>
    <material name="grid" texture="grid" texrepeat="8 8" reflectance="0.2"/>
  </asset>

  <!-- ====== World body ====== -->
  <worldbody>
    <!-- Ground plane -->
    <geom name="floor" type="plane" size="10 10 0.1"
          material="grid" condim="3"/>

    <!-- Lighting -->
    <light directional="true" diffuse="0.8 0.8 0.8" pos="0 0 4"
           dir="0 0 -1" castshadow="false"/>

    <!-- Terrain objects (stairs / slope / rocks injected here) -->
    {terrain_xml}

    <!-- ====== A1 torso (freejoint = 6-DOF floating base) ====== -->
    <body name="trunk" pos="0 0 0.42">
      <freejoint name="trunk_free"/>
      <site name="imu_site" pos="0 0 0" size="0.01"/>
      <geom type="box" size="0.1805 0.047 0.057"
            mass="{BODY_MASS}" rgba="0.2 0.4 0.7 1"/>
      {legs}
    </body>
  </worldbody>

  <!-- ====== Actuators (position servos, kp from robot_control.yaml) ====== -->
  <actuator>{actuators}
  </actuator>

  <!-- ====== Sensors ====== -->
  <sensor>{sensors}
  </sensor>

</mujoco>"""
    return xml


# Quick sanity check
import mujoco
test_xml = build_a1_xml()
test_model = mujoco.MjModel.from_xml_string(test_xml)
print('MJCF model loaded successfully!')
print(f'  nq={test_model.nq}  nv={test_model.nv}  nu={test_model.nu}')
print(f'  Joint names: {[test_model.joint(i).name for i in range(test_model.njnt)]}')

## 3. Procedural Terrain Generation

Four terrain types are implemented:
- **Flat** – plain ground (baseline)
- **Stairs** – geometry ported directly from `stairs.xacro` (length=0.64 m, width=0.31 m, height=0.17 m)
- **Slope** – inclined ramp
- **Rocks** – scattered random box obstacles

Each terrain function returns an XML string that is injected into the world body.

In [ ]:
import random

# ---------------------------------------------------------------------------
# Terrain builders – return MJCF XML fragment strings
# ---------------------------------------------------------------------------

def terrain_flat() -> str:
    """Empty terrain (only the default ground plane)."""
    return ''


def terrain_stairs(
    n_steps: int = 5,
    step_height: float = STAIR_HEIGHT,
    step_depth: float = STAIR_WIDTH,
    step_width: float = STAIR_LENGTH,
    start_x: float = 0.6,
) -> str:
    """
    Staircase ported from stairs.xacro.
    Each step is a box geom; steps stack upward in z and forward in x.

    Parameters match the xacro properties:
        stair_height = 0.170 m
        stair_width  = 0.310 m  (depth of step)
        stair_length = 0.640 m  (width of staircase)
    """
    geoms = []
    for i in range(n_steps):
        # Each successive step is deeper in x and higher in z
        x = start_x + i * step_depth
        z = step_height * (i + 1) / 2.0  # box centre at half height
        h = step_height * (i + 1)        # total height of the box stack
        geoms.append(
            f'<geom name="stair_{i}" type="box" '
            f'pos="{x:.3f} 0 {z:.3f}" '
            f'size="{step_depth/2:.3f} {step_width/2:.3f} {z:.3f}" '
            f'rgba="0.5 0.5 0.5 1" condim="3"/>'
        )
    return '\n    '.join(geoms)


def terrain_slope(
    slope_deg: float = 15.0,
    ramp_length: float = 2.0,
    ramp_width: float = 1.5,
    start_x: float = 0.5,
) -> str:
    """
    A single inclined ramp (box tilted by slope_deg around the y-axis).
    """
    angle_rad = np.deg2rad(slope_deg)
    # Euler angles: rotate around y-axis
    return (
        f'<geom name="ramp" type="box" '
        f'pos="{start_x + ramp_length/2:.3f} 0 {ramp_length/2*np.sin(angle_rad):.3f}" '
        f'euler="0 {-angle_rad:.4f} 0" '
        f'size="{ramp_length/2:.3f} {ramp_width/2:.3f} 0.02" '
        f'rgba="0.6 0.4 0.2 1" condim="3"/>'
    )


def terrain_rocks(
    n_rocks: int = 12,
    x_range=(0.4, 2.5),
    y_range=(-0.8, 0.8),
    max_height: float = 0.08,
    seed: int = 0,
) -> str:
    """
    Randomly placed low box obstacles (rocks / rubble).
    """
    rng = random.Random(seed)
    geoms = []
    for i in range(n_rocks):
        x = rng.uniform(*x_range)
        y = rng.uniform(*y_range)
        h = rng.uniform(0.02, max_height)
        sz_x = rng.uniform(0.05, 0.15)
        sz_y = rng.uniform(0.05, 0.15)
        geoms.append(
            f'<geom name="rock_{i}" type="box" '
            f'pos="{x:.3f} {y:.3f} {h:.3f}" '
            f'size="{sz_x:.3f} {sz_y:.3f} {h:.3f}" '
            f'rgba="0.4 0.3 0.2 1" condim="3"/>'
        )
    return '\n    '.join(geoms)


# Registry – used by the environment to sample terrain types
TERRAIN_BUILDERS = [
    terrain_flat,
    terrain_stairs,
    terrain_slope,
    terrain_rocks,
]

# Sanity check: build each terrain and load in MuJoCo
for builder in TERRAIN_BUILDERS:
    t_xml = builder()
    xml = build_a1_xml(terrain_xml=t_xml)
    m = mujoco.MjModel.from_xml_string(xml)
    print(f'{builder.__name__:25s}  →  ngeom={m.ngeom}')

## 4. Custom Gymnasium Environment

### Design choices
| Component | Choice | Rationale |
|---|---|---|
| **Observations** | Joint pos/vel (12+12), base orientation quat (4), base lin/ang velocity (3+3), previous action (12) = **46-dim** | Proprioceptive only; no contact flags or terrain maps |
| **Actions** | Δ joint position targets (12-dim, ±0.5 rad around default pose) | Smooth, matches PD servo convention from `robot_control.yaml` |
| **Reward** | Upright bonus + height bonus – fall penalty – action rate penalty – energy penalty | Encourages stable standing without encouraging specific gait |
| **Episode** | 10 s (500 control steps at 50 Hz; each step runs 10 × 2 ms physics steps) | Long enough to assess stabilization |
| **Reset** | Random terrain, randomized initial pose ± 30°, random push ± 0.5 m/s | Domain randomization per episode |

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import mujoco
import numpy as np


class A1StandEnv(gym.Env):
    """
    MuJoCo Gymnasium environment for Unitree A1 stand stabilization.

    Observations (46-dim, all proprioceptive):
        [0:12]  joint positions          (rad)
        [12:24] joint velocities         (rad/s)
        [24:28] trunk quaternion         (w, x, y, z)
        [28:31] trunk linear velocity    (m/s)
        [31:34] trunk angular velocity   (rad/s)
        [34:46] previous action          (rad, Δ from default)

    Actions (12-dim):
        Δ joint position targets clipped to ±0.5 rad from DEFAULT_POSE.
        Scaled to [-1, 1] for the policy network.

    Domain randomization (applied every reset):
        - Terrain type (flat / stairs / slope / rocks)
        - Surface friction (uniform [0.5, 1.5])
        - Trunk mass perturbation (± 20 %)
        - Initial joint noise (± 0.3 rad)
        - Initial trunk tilt (± 0.5 rad in roll/pitch)
        - Random velocity disturbance (± 0.5 m/s)
        - Actuator output noise (added each step, σ = 0.01)
    """

    metadata = {'render_modes': ['rgb_array'], 'render_fps': 50}

    # --- Environment constants ---
    DT           = 0.002          # MuJoCo simulation timestep (s)
    CONTROL_HZ   = 50             # Policy control frequency (Hz)
    SIM_STEPS    = round(1.0 / (DT * CONTROL_HZ))    # sim steps per control step (10 @ 50 Hz)
    MAX_EPISODE_STEPS = 500       # = 10 s at 50 Hz

    # Action scale: Δ target in radians
    ACTION_SCALE = 0.5

    # Observation bounds (generous)
    OBS_HIGH = np.array(
        [np.pi] * 12 +       # joint pos
        [30.0]  * 12 +       # joint vel
        [1.0]   *  4 +       # quaternion
        [5.0]   *  3 +       # lin vel
        [10.0]  *  3 +       # ang vel
        [1.0]   * 12,        # prev action
        dtype=np.float32
    )

    def __init__(self, render_mode=None, terrain_type=None, seed=None):
        """
        Parameters
        ----------
        terrain_type : int or None
            Index into TERRAIN_BUILDERS list. None = random each episode.
        """
        super().__init__()
        self.render_mode  = render_mode
        self.terrain_type = terrain_type  # None = randomised
        self._rng         = np.random.default_rng(seed)

        # Build initial model (flat terrain)
        self._load_model(terrain_xml=terrain_flat())

        # Gymnasium spaces
        self.observation_space = spaces.Box(
            low=-self.OBS_HIGH, high=self.OBS_HIGH, dtype=np.float32
        )
        self.action_space = spaces.Box(
            low=-1.0, high=1.0, shape=(12,), dtype=np.float32
        )

        self._prev_action = np.zeros(12, dtype=np.float32)
        self._step_count  = 0

    # ------------------------------------------------------------------
    # Private helpers
    # ------------------------------------------------------------------

    def _load_model(self, terrain_xml: str):
        """(Re)build MuJoCo model + data with the given terrain XML."""
        xml = build_a1_xml(terrain_xml=terrain_xml)
        self.model = mujoco.MjModel.from_xml_string(xml)
        self.data  = mujoco.MjData(self.model)
        # Cache actuator and sensor ids
        self._n_joints = 12
        # Joint qpos offset: freejoint adds 7 DOF (pos3 + quat4) before the 12 hinge joints
        self._jpos_start = 7
        self._jvel_start = 6   # freejoint has 6 vel dofs

    def _get_obs(self) -> np.ndarray:
        d = self.data
        joint_pos  = d.qpos[self._jpos_start:self._jpos_start + 12].astype(np.float32)
        joint_vel  = d.qvel[self._jvel_start:self._jvel_start + 12].astype(np.float32)
        trunk_quat = d.qpos[3:7].astype(np.float32)   # w x y z order in MuJoCo
        trunk_lvel = d.qvel[0:3].astype(np.float32)
        trunk_avel = d.qvel[3:6].astype(np.float32)
        obs = np.concatenate([
            joint_pos, joint_vel, trunk_quat,
            trunk_lvel, trunk_avel, self._prev_action
        ])
        return np.clip(obs, -self.OBS_HIGH, self.OBS_HIGH)

    def _compute_reward(self) -> tuple[float, dict]:
        d = self.data

        # --- Upright bonus: reward quat alignment with [1,0,0,0] (z-up) ---
        quat = d.qpos[3:7]            # w x y z
        # cos^2(θ/2) ≈ quat[0]^2 when no yaw; penalise roll/pitch tilt
        uprightness = float(quat[0] ** 2)   # 1 when upright, <1 when tilted

        # --- Height bonus: reward staying at ~0.42 m (nominal standing height) ---
        trunk_z = float(d.qpos[2])
        target_z = 0.42
        height_reward = np.exp(-5.0 * (trunk_z - target_z) ** 2)

        # --- Action rate penalty: smooth control ---
        action_rate = float(np.sum(self._prev_action ** 2))

        # --- Energy penalty: penalize large joint velocities × torques ---
        joint_vel  = d.qvel[self._jvel_start:self._jvel_start + 12]
        joint_ctrl = d.ctrl[:12]
        energy = float(np.sum(np.abs(joint_vel * joint_ctrl)))

        # --- Fall detection ---
        fell = trunk_z < 0.2

        # Composite reward
        reward = (
            2.0 * uprightness
            + 1.0 * height_reward
            - 0.01 * action_rate
            - 0.001 * energy
            - 50.0 * float(fell)
        )
        info = {
            'uprightness': uprightness,
            'height_reward': height_reward,
            'trunk_z': trunk_z,
            'fell': fell,
        }
        return float(reward), info

    # ------------------------------------------------------------------
    # Domain randomization (applied at every reset)
    # ------------------------------------------------------------------

    def _apply_domain_randomization(self):
        rng = self._rng

        # 1. Friction randomization
        # Note: _load_model() creates a fresh MjModel each episode, so friction
        # values are always reset to MJCF defaults before this scale is applied.
        friction_scale = rng.uniform(0.5, 1.5)
        self.model.geom_friction[:, 0] *= friction_scale

        # 2. Trunk mass perturbation (±20 %)
        # Note: mass is reset to MJCF defaults each episode by _load_model(),
        # so this scale is applied exactly once per episode without compounding.
        trunk_body_id = mujoco.mj_name2id(
            self.model, mujoco.mjtObj.mjOBJ_BODY, 'trunk'
        )
        mass_scale = rng.uniform(0.8, 1.2)
        self.model.body_mass[trunk_body_id] *= mass_scale

        # 3. Random initial trunk orientation (±0.5 rad roll/pitch)
        roll  = rng.uniform(-0.5, 0.5)
        pitch = rng.uniform(-0.3, 0.3)
        # Convert roll/pitch to quaternion.
        # Yaw is kept at 0: we don't randomize the robot's heading direction.
        cy = np.cos(0 / 2); sy = np.sin(0 / 2)
        cp = np.cos(pitch / 2); sp = np.sin(pitch / 2)
        cr = np.cos(roll  / 2); sr = np.sin(roll  / 2)
        qw = cr * cp * cy + sr * sp * sy
        qx = sr * cp * cy - cr * sp * sy
        qy = cr * sp * cy + sr * cp * sy
        qz = cr * cp * sy - sr * sp * cy
        self.data.qpos[3:7] = [qw, qx, qy, qz]

        # 4. Random initial joint positions (±0.3 rad from default)
        joint_noise = rng.uniform(-0.3, 0.3, size=12)
        self.data.qpos[self._jpos_start:self._jpos_start + 12] = (
            DEFAULT_POSE + joint_noise
        )

        # 5. Random velocity push (±0.5 m/s linear)
        self.data.qvel[0:3] = rng.uniform(-0.5, 0.5, size=3)

    # ------------------------------------------------------------------
    # Gymnasium API
    # ------------------------------------------------------------------

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self._rng = np.random.default_rng(seed)

        # 1. Pick terrain type
        idx = (
            self.terrain_type
            if self.terrain_type is not None
            else int(self._rng.integers(len(TERRAIN_BUILDERS)))
        )
        terrain_fn = TERRAIN_BUILDERS[idx]
        self._load_model(terrain_xml=terrain_fn())

        # 2. Initialise to default standing pose
        mujoco.mj_resetData(self.model, self.data)
        self.data.qpos[2] = 0.42   # trunk height
        self.data.qpos[self._jpos_start:self._jpos_start + 12] = DEFAULT_POSE

        # 3. Domain randomization
        self._apply_domain_randomization()

        mujoco.mj_forward(self.model, self.data)

        self._prev_action = np.zeros(12, dtype=np.float32)
        self._step_count  = 0

        return self._get_obs(), {}

    def step(self, action: np.ndarray):
        action = np.clip(action, -1.0, 1.0).astype(np.float32)

        # Add actuator noise (domain randomization, σ=0.01)
        action_noisy = action + self._rng.normal(0, 0.01, size=12).astype(np.float32)

        # Map normalised action → joint position targets
        target_pos = DEFAULT_POSE + action_noisy * self.ACTION_SCALE
        self.data.ctrl[:12] = target_pos

        # Simulate SIM_STEPS physics steps per control step (10 × 2 ms = 20 ms @ 50 Hz)
        for _ in range(self.SIM_STEPS):
            mujoco.mj_step(self.model, self.data)

        self._prev_action = action
        self._step_count += 1

        obs            = self._get_obs()
        reward, info   = self._compute_reward()
        terminated     = bool(info['fell'])
        truncated      = self._step_count >= self.MAX_EPISODE_STEPS

        return obs, reward, terminated, truncated, info

    def render(self):
        if self.render_mode != 'rgb_array':
            return None
        try:
            renderer = mujoco.Renderer(self.model, height=480, width=640)
            renderer.update_scene(self.data, camera=-1)
            frame = renderer.render()
            renderer.close()
            return frame
        except Exception:
            # Rendering may fail in headless environments without a display server.
            # Install libosmesa6-dev and set MUJOCO_GL=osmesa for offscreen rendering.
            return None

    def close(self):
        pass


# --- Quick smoke test ---
env = A1StandEnv()
obs, _ = env.reset(seed=42)
print('Observation shape:', obs.shape)
print('Action space:',      env.action_space)
for _ in range(5):
    obs, rew, term, trunc, info = env.step(env.action_space.sample())
print(f'Sample step  reward={rew:.3f}  trunk_z={info["trunk_z"]:.3f}  fell={info["fell"]}')
env.close()

## 5. Register and verify with Gymnasium's `check_env`

In [ ]:
from stable_baselines3.common.env_checker import check_env

env = A1StandEnv()
check_env(env, warn=True)
print('Environment check passed!')
env.close()

## 6. RL Setup – PPO with Stable Baselines3

We use **PPO** (Proximal Policy Optimization) with:
- **MLP policy** (two hidden layers of 256 units each)
- **Vectorised environments** (`SubprocVecEnv`) for parallel rollout collection
- **VecNormalize** for automatic observation and reward normalisation

> **SAC alternative:** Just swap `PPO` for `SAC` and remove `n_steps`.
> SAC is off-policy so it can reuse older transitions, but requires more tuning.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecNormalize, SubprocVecEnv
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
import os

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
N_ENVS        = 4          # parallel rollout environments
TOTAL_STEPS   = 500_000    # total environment interactions (increase for better convergence)
LOG_DIR       = './rl_logs'
MODEL_DIR     = './rl_models'
os.makedirs(LOG_DIR,   exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# Build vectorised training environments
# ---------------------------------------------------------------------------
# Build per-environment callables so each subprocess gets a unique seed
env_fns = [lambda rank=r: A1StandEnv(seed=rank) for r in range(N_ENVS)]

train_env = SubprocVecEnv(env_fns)
train_env = VecNormalize(train_env, norm_obs=True, norm_reward=True, clip_obs=10.0)

# ---------------------------------------------------------------------------
# Evaluation environment (flat terrain, deterministic reset)
# ---------------------------------------------------------------------------
eval_env = make_vec_env(lambda: A1StandEnv(terrain_type=0, seed=999),
                        n_envs=1, seed=999)
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False,
                        training=False, clip_obs=10.0)

# ---------------------------------------------------------------------------
# PPO model
# ---------------------------------------------------------------------------
model = PPO(
    policy            = 'MlpPolicy',
    env               = train_env,
    learning_rate     = 3e-4,
    n_steps           = 2048,          # rollout length per env
    batch_size        = 512,
    n_epochs          = 10,
    gamma             = 0.99,
    gae_lambda        = 0.95,
    clip_range        = 0.2,
    ent_coef          = 0.005,
    vf_coef           = 0.5,
    max_grad_norm     = 0.5,
    policy_kwargs     = dict(net_arch=[256, 256]),
    tensorboard_log   = LOG_DIR,
    verbose           = 1,
    seed              = 0,
)

print(model.policy)
print(f'\nTraining for {TOTAL_STEPS:,} steps across {N_ENVS} parallel envs.')

# ---------------------------------------------------------------------------
# SAC alternative (uncomment to use instead of PPO)
# ---------------------------------------------------------------------------
# from stable_baselines3 import SAC
# model = SAC(
#     policy         = 'MlpPolicy',
#     env            = train_env,
#     learning_rate  = 3e-4,
#     buffer_size    = 500_000,
#     batch_size     = 256,
#     gamma          = 0.99,
#     tau            = 0.005,
#     train_freq     = 1,
#     gradient_steps = 1,
#     ent_coef       = 'auto',
#     policy_kwargs  = dict(net_arch=[256, 256]),
#     tensorboard_log= LOG_DIR,
#     verbose        = 1,
# )

## 7. Training Loop (Jupyter-friendly)

Callbacks:
- **`EvalCallback`** – evaluates the policy every 50 k steps, saves the best model
- **`CheckpointCallback`** – saves periodic snapshots

You can interrupt training at any time; the best model is preserved.

In [ ]:
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path = os.path.join(MODEL_DIR, 'best'),
    log_path             = LOG_DIR,
    eval_freq            = 50_000 // N_ENVS,
    n_eval_episodes      = 10,
    deterministic        = True,
    render               = False,
)

checkpoint_callback = CheckpointCallback(
    save_freq  = 100_000 // N_ENVS,
    save_path  = MODEL_DIR,
    name_prefix= 'a1_stand',
)

# --- Train! ---
model.learn(
    total_timesteps  = TOTAL_STEPS,
    callback         = [eval_callback, checkpoint_callback],
    progress_bar     = True,    # rich progress bar in Jupyter
    reset_num_timesteps = True,
)

# Save final model and normalisation statistics
model.save(os.path.join(MODEL_DIR, 'a1_stand_final'))
train_env.save(os.path.join(MODEL_DIR, 'vec_normalize_final.pkl'))
print('Training complete. Model saved.')

## 8. Learning Curve Visualisation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

# SB3 EvalCallback saves evaluations.npz in the log_path
eval_results_file = os.path.join(LOG_DIR, 'evaluations.npz')

if os.path.exists(eval_results_file):
    data = np.load(eval_results_file)
    timesteps    = data['timesteps']
    ep_rewards   = data['results']          # shape (n_evals, n_eval_episodes)
    ep_lengths   = data['ep_lengths']

    mean_rew = ep_rewards.mean(axis=1)
    std_rew  = ep_rewards.std(axis=1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(timesteps, mean_rew, label='Mean reward')
    axes[0].fill_between(timesteps,
                         mean_rew - std_rew,
                         mean_rew + std_rew, alpha=0.3)
    axes[0].set_xlabel('Environment steps')
    axes[0].set_ylabel('Episode reward')
    axes[0].set_title('Evaluation reward over training')
    axes[0].legend()

    mean_len = ep_lengths.mean(axis=1)
    axes[1].plot(timesteps, mean_len, color='orange', label='Mean ep length')
    axes[1].set_xlabel('Environment steps')
    axes[1].set_ylabel('Episode length (steps)')
    axes[1].set_title('Episode length over training')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('learning_curve.png', dpi=150)
    plt.show()
    print('Learning curve saved to learning_curve.png')
else:
    print('No evaluation results found yet. Run the training cell first.')

## 9. Load best model and run evaluation

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecNormalize, DummyVecEnv
import numpy as np

# Load model and normalisation statistics
best_model_path = os.path.join(MODEL_DIR, 'best', 'best_model')
norm_stats_path = os.path.join(MODEL_DIR, 'vec_normalize_final.pkl')

# Build a single-environment evaluation wrapper
def make_eval_env(terrain_idx: int = 0):
    env = DummyVecEnv([lambda: A1StandEnv(
        render_mode='rgb_array',
        terrain_type=terrain_idx,
        seed=123
    )])
    if os.path.exists(norm_stats_path):
        env = VecNormalize.load(norm_stats_path, env)
        env.training = False
        env.norm_reward = False
    return env

if os.path.exists(best_model_path + '.zip'):
    loaded_model = PPO.load(best_model_path)
    print('Best model loaded.')
else:
    # Fallback: use the model object from training
    loaded_model = model
    print('Using model from training session (best_model not found on disk).')

TERRAIN_NAMES = ['Flat', 'Stairs', 'Slope', 'Rocks']

print('\n=== Evaluation across all terrain types ===')
for t_idx, t_name in enumerate(TERRAIN_NAMES):
    ev = make_eval_env(terrain_idx=t_idx)
    obs = ev.reset()
    ep_rewards, ep_lengths, ep_fell = [], [], []
    n_eval = 5
    for ep in range(n_eval):
        done = False
        ep_rew, ep_len, fell = 0.0, 0, False
        obs = ev.reset()
        while not done:
            action, _ = loaded_model.predict(obs, deterministic=True)
            obs, rew, done, info = ev.step(action)
            ep_rew += float(rew[0])
            ep_len += 1
            if info[0].get('fell', False):
                fell = True
        ep_rewards.append(ep_rew)
        ep_lengths.append(ep_len)
        ep_fell.append(fell)
    ev.close()
    success_rate = 1.0 - np.mean(ep_fell)
    print(f'  {t_name:8s}  reward={np.mean(ep_rewards):7.1f}±{np.std(ep_rewards):.1f}'
          f'  ep_len={np.mean(ep_lengths):5.0f}  success={success_rate:.0%}')

## 10. Video Recording

Renders the trained policy on each terrain type and saves an MP4 video.
Uses `imageio` (FFmpeg backend) to encode frames collected from `render()`.

In [ ]:
import imageio
from IPython.display import Video, display

VIDEO_DIR = './videos'
os.makedirs(VIDEO_DIR, exist_ok=True)


def record_episode(
    policy,
    terrain_idx: int,
    max_steps: int = 500,
    fps: int = 25,
    seed: int = 0,
) -> str:
    """
    Run one episode, save frames, write MP4, return file path.

    Parameters
    ----------
    policy      : callable(obs) -> action  (any SB3 model with .predict)
    terrain_idx : index into TERRAIN_BUILDERS
    max_steps   : maximum episode length
    fps         : output video frame rate
    seed        : environment seed for reproducibility
    """
    env = A1StandEnv(
        render_mode  = 'rgb_array',
        terrain_type = terrain_idx,
        seed         = seed,
    )

    obs, _ = env.reset(seed=seed)
    frames  = []
    total_r = 0.0

    for step in range(max_steps):
        # SB3 expects a (1, obs_dim) array when using VecNormalize; here we use
        # the raw env, so reshape accordingly.
        action, _ = policy.predict(obs[np.newaxis, :], deterministic=True)
        action = action[0]  # unwrap batch dim

        obs, rew, terminated, truncated, info = env.step(action)
        total_r += rew

        frame = env.render()
        if frame is not None:
            frames.append(frame)

        if terminated or truncated:
            break

    env.close()

    t_name   = TERRAIN_NAMES[terrain_idx]
    out_path = os.path.join(VIDEO_DIR, f'a1_stand_{t_name.lower()}.mp4')
    imageio.mimsave(out_path, frames, fps=fps, quality=5)
    print(f'  Saved {t_name} video: {out_path}  '
          f'({len(frames)} frames, total_reward={total_r:.1f})')
    return out_path


# Record a video for each terrain type
print('Recording evaluation videos...')
video_paths = []
for t_idx, t_name in enumerate(TERRAIN_NAMES):
    path = record_episode(
        policy      = loaded_model,
        terrain_idx = t_idx,
        max_steps   = 250,
        fps         = 25,
        seed        = t_idx * 7,
    )
    video_paths.append(path)

print('\nAll videos saved.')

In [ ]:
# Display the 'flat' terrain video inline
if video_paths:
    display(Video(video_paths[0], embed=True, width=640))

## 11. Summary — Reference repo asset mapping

| `gmaymon3/robot_quadruped_mpc_controller` file | Status in this notebook | Notes |
|---|---|---|
| `a1_description/urdf/a1.urdf` | ✅ **Adapted** | Joint structure, link names, masses extracted and converted to MJCF in `build_a1_xml()` |
| `a1_description/xacro/stairs.xacro` | ✅ **Adapted** | Stair dimensions (length=0.64, width=0.31, height=0.17 m) reproduced in `terrain_stairs()` |
| `a1_description/config/robot_control.yaml` | ✅ **Adapted** | PD gains (hip P=100 D=5, thigh/calf P=300 D=8) used as MuJoCo `kp` / joint damping |
| `a1_description/xacro/gazebo.xacro` | ⚠️ **Concept reused** | IMU, contact sensors replaced by MuJoCo sensor API + `mjData` reads |
| `a1_description/xacro/leg.xacro` | ✅ **Adapted** | 3-DOF leg (hip-thigh-calf) × 4 reproduced in `leg_xml()` |
| `a1_description/xacro/robot.xacro` | ✅ **Adapted** | Robot assembly logic reproduced in `build_a1_xml()` |
| `a1_description/xacro/transmission.xacro` | ❌ **Not reused** | ROS transmission definitions replaced by MuJoCo position actuators |
| `slprj/` (MATLAB Simulink) | ❌ **Not reused** | MATLAB MPC replaced entirely by PPO/SAC neural network policy |
| `a1_description/launch/` | ❌ **Not reused** | ROS launch files not relevant in MuJoCo native workflow |

### What was fully rewritten for MuJoCo + RL
1. **Physics model** – URDF/Xacro → MJCF XML with capsule geoms (mesh-free, portable)
2. **Terrain generation** – Xacro macros → Python functions returning MJCF geom strings
3. **Controller** – MATLAB MPC → deep RL policy (PPO / SAC via SB3)
4. **Sensor interface** – Gazebo ROS plugins → MuJoCo `sensor` elements + `mjData` reads
5. **Simulation backend** – Gazebo → native MuJoCo Python API
6. **Domain randomization** – none in reference repo → per-reset terrain/friction/mass/pose randomisation

### Key metric definitions
| Metric | Definition |
|---|---|
| **Success rate** | Fraction of episodes where `trunk_z > 0.2` until `MAX_EPISODE_STEPS` |
| **Time-to-stabilize** | Episode step at which upright score > 0.95 for 50 consecutive steps |
| **Robustness** | Mean success rate across all 4 terrain types with randomised friction/mass |
